[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/meta_analysis_Practical.ipynb)

# 🧬 Practical: Meta-analysis of GWAS summary statistics

Welcome! In this practical we build the core tools of GWAS **meta-analysis** *from scratch* in Python, so you understand what's happening under the hood rather than treating the software as a black box.

We will cover:

1. **Fixed-effects inverse-variance weighting (IVW)** — the workhorse for combining effect-size estimates, and *why* inverse-variance weights are special.
2. **Combining p-values** — Stouffer's Z and the **Cauchy combination test**, including why Cauchy stays well-behaved when tests are *correlated*.
3. **A real-data capstone** — meta-analysing GIANT consortium GWAS summary statistics.
4. **Bonus challenges** — random-effects meta-analysis, confidence-interval coverage, and a power comparison.

**Why meta-analyse?**
- 📈 **Bigger effective sample size** → more power to detect true associations.
- 🔒 **You often can't pool individual-level data** (privacy, consent, governance). Summary statistics are shareable, so meta-analysis is how consortia like GIANT, PGC and GWAS Catalog combine cohorts.
- 🧪 **Combining p-values** across correlated tests can reduce the multiple-testing burden.

---

## 🛠️ 1. Environment Setup

This is a **Python** notebook — no R magic this time. We only need the standard scientific stack, which is pre-installed on Colab. We fix a random seed so everyone gets the same numbers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(2026)   # fixed seed for reproducibility
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True
print("Setup complete ✅")

## 🧮 2. Fixed-effects inverse-variance weighting (IVW)

A **fixed-effects** meta-analysis assumes every study is estimating the *same single true effect* $\beta$. Any differences between studies are just **sampling noise**.

Study $k$ reports an effect estimate $\hat\beta_k$ and its standard error $\text{se}_k$. We model

$$\hat\beta_k \sim \mathrm{N}(\beta,\; \text{se}_k^2).$$

We want to combine the $\hat\beta_k$ into a single pooled estimate. A natural idea is a **weighted average**:

$$\hat\beta_{\text{pooled}} = \frac{\sum_k w_k \hat\beta_k}{\sum_k w_k}.$$

> **A note on indexing.** In the lecture each estimate carried *two* subscripts — one for the study and one for the SNP, e.g. $\hat\beta_{kj}$ for study $k$ at SNP $j$. From here on we keep things simple by following a **single** SNP at a time, so we drop the SNP index $j$ and just write $\hat\beta_k$ for "study $k$'s estimate".

The question is: *what weights should we use?* Let's simulate some studies and find out.

In [ ]:
# Simulate K studies all estimating the SAME true effect beta
beta_true = 0.30                              # the single true effect size we want to recover
K = 8                                         # number of studies in the meta-analysis

# Studies differ in precision: standard errors vary (e.g. different sample sizes)
se = rng.uniform(0.05, 0.30, size=K)          # each study's standard error (small se = precise study)
beta_hat = rng.normal(beta_true, se)          # each study's noisy effect estimate

studies = pd.DataFrame({
    "study":    [f"Study {i+1}" for i in range(K)],
    "beta_hat": beta_hat,
    "se":       se,
})
studies

### ❓ Question 1: Which studies should count for more?

We'll combine the studies as a weighted average $\hat\beta_{\text{pooled}} = \sum_k w_k\hat\beta_k / \sum_k w_k$.

**Think first (this is the real question):** which studies deserve a *bigger* weight $w_k$ — and why? A study with a tiny standard error is very precise, so we'd like to trust it more. The variance of study $k$ is $\text{se}_k^2$, so weighting by the **inverse variance** $w_k = 1/\text{se}_k^2$ does exactly that.

Now fill in the three `...` lines in `ivw()` below. The comments tell you which NumPy pieces to use and how to combine them — you're assembling the idea, not writing clever code. (The pooled SE is $1/\sqrt{\sum_k w_k}$.)

<details>
<summary>💡 Click here to reveal the answer!</summary>

```python
def ivw(beta_hat, se):
    w = 1.0 / se**2                          # inverse-variance weights
    beta_pooled = np.sum(w * beta_hat) / np.sum(w)   # weighted average
    se_pooled   = np.sqrt(1.0 / np.sum(w))   # pooled SE = 1/sqrt(sum of weights)
    return beta_pooled, se_pooled
```

Smaller-SE (more precise) studies get larger weights, so they pull the pooled estimate towards themselves. The pooled SE is always **smaller** than any individual study's SE — that's the power gain from meta-analysis.

</details>

In [ ]:
def ivw(beta_hat, se):
    # 1) A study's variance is its se**2. Build weights that REWARD precision
    #    (small se -> big weight).
    w = ...               # hint: weight = one over the variance

    # 2) Pooled estimate = the WEIGHTED AVERAGE of the estimates.
    beta_pooled = ...     # hint: a weighted average is sum(w*x) divided by sum(w)

    # 3) The pooled variance is 1 / (total weight); the SE is its square root.
    se_pooled   = ...     # hint: combine np.sqrt, np.sum and the weights

    return beta_pooled, se_pooled

# --- once you've filled in ivw(), this cell will run ---
beta_ivw, se_ivw = ivw(studies.beta_hat.values, studies.se.values)
beta_unweighted  = studies.beta_hat.mean()

print(f"True effect            : {beta_true:.3f}")
print(f"Unweighted mean        : {beta_unweighted:.3f}")
print(f"IVW pooled estimate    : {beta_ivw:.3f}  (SE = {se_ivw:.3f})")
print(f"Smallest single-study SE: {studies.se.min():.3f}  <-- IVW SE is smaller still")

<details>
<summary>✅ Reveal the expected output</summary>

```
True effect            : 0.300
Unweighted mean        : 0.310
IVW pooled estimate    : 0.293  (SE = 0.049)
Smallest single-study SE: 0.094
```

The IVW estimate (0.293) sits close to the true 0.30, and crucially its SE (0.049) is **smaller than even the most precise single study** (0.094). That shrinking SE is the power you gain by pooling. (Exact digits come from the fixed seed; the *pattern* is what matters.)

</details>

### ❓ Question 2: Why inverse-variance weights? (the key idea)

Notice the weighted average $\hat\beta_{\text{pooled}} = \sum_k w_k\hat\beta_k / \sum_k w_k$ is **unbiased for *any* choice of weights** (as long as they don't depend on the data). So why pick $w_k = 1/\text{se}_k^2$ specifically?

**Claim: inverse-variance weights give the pooled estimator with the *smallest possible variance*.**

Let's check this *empirically* with a Monte-Carlo simulation. We'll repeatedly simulate the same set of studies and measure the spread (variance) of the pooled estimate under four weighting schemes:

| Scheme | Weight $w_k$ |
|---|---|
| Uniform | $1$ |
| ∝ sample size | $N_k$ |
| ∝ 1/se | $1/\text{se}_k$ |
| **IVW** | $1/\text{se}_k^2$ |

Because a study's variance scales as $\text{se}_k^2 \propto 1/N_k$, weighting by sample size $w_k = N_k$ is *identical* to IVW — which is why those two schemes come out on top together below.

<details>
<summary>💡 Click here to reveal the answer!</summary>

The IVW scheme (∝ $1/\text{se}_k^2$) produces the **lowest empirical variance** of the pooled estimate. This is the Gauss–Markov / inverse-variance result: among all unbiased weighted averages, weighting by inverse variance is the [best linear unbiased estimator](https://en.wikipedia.org/wiki/Inverse-variance_weighting).

(Note "∝ sample size" coincides with IVW here because under a simple model $\text{se}_k^2 \propto 1/N_k$, so $N_k \propto 1/\text{se}_k^2$. That's exactly *why* sample-size weighting works so well in practice.)

</details>

In [ ]:
# Keep the study SEs fixed; redraw the noisy estimates many times
n_rep = 20000                                 # number of Monte-Carlo replicates
fixed_se = studies.se.values                  # the K standard errors, held constant across replicates

schemes = {
    "uniform":      np.ones_like(fixed_se),
    "prop N (~1/se^2)": 1.0 / fixed_se**2,   # sample size ~ 1/se^2
    "prop 1/se":    1.0 / fixed_se,
    "IVW (1/se^2)": 1.0 / fixed_se**2,
}

# Draw all replicates at once: shape (n_rep, K)
draws = rng.normal(beta_true, fixed_se, size=(n_rep, len(fixed_se)))

emp_var = {}                                     # empirical variance of the pooled estimate, per scheme
for name, w in schemes.items():                  # w = the weight vector for this scheme
    pooled = (draws * w).sum(axis=1) / w.sum()   # weighted average per replicate
    emp_var[name] = pooled.var()                 # spread of the estimate -> lower is better

for name, v in emp_var.items():
    print(f"{name:18s} empirical variance = {v:.3e}")

plt.bar(list(emp_var.keys()), list(emp_var.values()), color=["#bbb","#88c","#8c8","#c44"])
plt.ylabel("Empirical variance of pooled estimate")
plt.title("IVW minimises the variance of the pooled estimate")
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

<details>
<summary>✅ Reveal the expected output</summary>

Empirical variance of the pooled estimate (lower = better):

```
uniform            empirical variance = 4.17e-03
prop N (~1/se^2)   empirical variance = 2.43e-03
prop 1/se          empirical variance = 2.77e-03
IVW (1/se^2)       empirical variance = 2.43e-03
```

**IVW — and the equivalent sample-size weighting — give the smallest variance.** Uniform weighting is ~1.7× worse, and even the sensible-looking $1/\text{se}_k$ scheme loses out. No weighting beats inverse variance: that is the inverse-variance / Gauss–Markov result, seen here empirically.

</details>

## 🎲 3. Combining p-values: Stouffer's Z and the Cauchy combination test

Sometimes we don't have comparable effect sizes — only **p-values** (and maybe a direction and sample size). We then want to combine the p-values into a single test of "is there a signal *anywhere*?". This also lets us **reduce the multiple-testing burden**: one combined test instead of many.

We'll look at two methods:
- **Stouffer's Z** — convert each p-value to a Z-score and average.
- **Cauchy combination test (ACAT)** — transform each p-value through a tangent function.

Then we'll see the crucial difference between them when the tests are **correlated**.

### ❓ Question 3: Stouffer's Z — averaging evidence on the Z-scale

**The intuition:** a p-value is awkward to average directly, but the matching **Z-score** isn't. So we (1) turn each p into a Z-score, (2) take a weighted average of the Z's, scaled so the result is again standard normal under the null, and (3) turn that back into a p-value. Bigger studies should count more, so weights are often $w_k \propto \sqrt{N_k}$:

$$Z_{\text{Stouffer}} = \frac{\sum_k w_k Z_k}{\sqrt{\sum_k w_k^2}}, \qquad Z_k = \Phi^{-1}(1-p_k), \qquad p = 1 - \Phi(Z_{\text{Stouffer}}).$$

Fill in the three `...` lines using the hints. (`stats.norm.isf` is $\Phi^{-1}(1-\cdot)$ and `stats.norm.sf` is $1-\Phi(\cdot)$.)

<details>
<summary>💡 Click here to reveal the answer!</summary>

```python
def stouffer(p, w=None):
    z = stats.norm.isf(p)              # Phi^{-1}(1 - p)
    if w is None:
        w = np.ones_like(p)
    z_comb = np.sum(w * z) / np.sqrt(np.sum(w**2))
    return stats.norm.sf(z_comb)       # 1 - Phi(z_comb)
```

Under the null, the combined Z is standard normal because a weighted sum of independent normals is normal, and we divide by exactly its standard deviation. The $\sqrt{N_k}$ weighting is the p-value analogue of inverse-variance weighting.

</details>

In [ ]:
def stouffer(p, w=None):
    p = np.asarray(p, float)
    w = np.ones_like(p) if w is None else np.asarray(w, float)

    # 1) Turn each one-sided p-value into a Z-score, Phi^{-1}(1 - p).
    z = ...          # hint: which stats.norm method does this in one step? (look at stats.norm.isf)

    # 2) Combine the z's. Use a weighted sum, but divide by sqrt(sum of w**2)
    #    (NOT sum of w) so the result is standard normal under the null.
    z_comb = ...     # hint: assemble it from np.sum and np.sqrt

    # 3) Turn the combined Z back into an upper-tail p-value, 1 - Phi(z_comb).
    return ...       # hint: stats.norm.sf ("survival function") gives 1 - Phi(.)

# once filled in: combining several modest p-values should give a smaller one
print("Stouffer of [0.04, 0.03, 0.06, 0.05] =", f"{stouffer([0.04,0.03,0.06,0.05]):.4g}")

<details>
<summary>✅ Reveal the expected output</summary>

```
Stouffer of [0.04, 0.03, 0.06, 0.05] = 0.0003183
```

Four individually-modest p-values (each ~0.03–0.06) combine to $\approx 3\times10^{-4}$. None of them alone would survive multiple-testing correction, but stacking several *concordant* weak signals yields one decisively stronger one — the whole point of combining evidence.

</details>

### ❓ Question 4: The Cauchy combination test

The Cauchy combination test (ACAT) builds the statistic

$$T = \sum_k w_k \tan\!\big((0.5 - p_k)\,\pi\big), \qquad \sum_k w_k = 1,\; w_k \ge 0.$$

The magic: if $p_k \sim \text{Uniform}(0,1)$ under the null, then each $\tan((0.5-p_k)\pi)$ is a **standard Cauchy** random variable, and a weighted sum of Cauchys is *again Cauchy*. So the null distribution of $T$ is standard Cauchy regardless of $K$, and the p-value is

$$p_{\text{Cauchy}} = 0.5 - \frac{\arctan(T)}{\pi}.$$

**The intuition to hold onto:** the tangent transform sends a small $p$ to a *huge* number, so one strong signal dominates $T$ — and the Cauchy null never changes shape no matter how the tests are correlated (that's Question 6). Fill in the three `...` lines.

<details>
<summary>💡 Click here to reveal the answer!</summary>

```python
def cauchy_combination(p, w=None):
    p = np.asarray(p, float)
    if w is None:
        w = np.ones_like(p)
    w = w / w.sum()                         # weights must sum to 1
    T = np.sum(w * np.tan((0.5 - p) * np.pi))
    return 0.5 - np.arctan(T) / np.pi       # standard-Cauchy survival function
```

For very small $p$, $\tan((0.5-p)\pi) = \cot(\pi p) \approx 1/(\pi p)$, which **blows up**. So a single tiny p-value dominates the sum — exactly what we want from a test for "is anything significant?".

</details>

In [ ]:
def cauchy_combination(p, w=None):
    p = np.asarray(p, float)
    w = np.ones_like(p) if w is None else np.asarray(w, float)
    w = w / w.sum()                              # weights must sum to 1

    # 1) Map each p-value through the tangent transform in the formula above
    #    (it turns a uniform null p into a standard Cauchy draw).
    terms = ...      # hint: use np.tan on the (0.5 - p) * np.pi argument (pi is np.pi)

    # 2) Build the statistic T as a weighted sum of those terms.
    T = ...          # hint: combine w and terms with np.sum

    # 3) Convert T to a p-value using the standard-Cauchy tail from the formula above.
    return ...       # hint: it's 0.5 minus np.arctan(T) divided by np.pi

print("Cauchy of [0.04, 0.03, 0.06, 0.05] =", f"{cauchy_combination([0.04,0.03,0.06,0.05]):.4g}")

<details>
<summary>✅ Reveal the expected output</summary>

```
Cauchy of [0.04, 0.03, 0.06, 0.05] = 0.04212
```

With **no single dominant signal**, Cauchy returns ~0.042 — only just below 0.05. Contrast this with Stouffer's $\approx 3\times10^{-4}$ on the *same* four p-values: Stouffer rewards *many concordant* moderate signals, while Cauchy is driven by the *single strongest* one (and here there isn't a strong one). They are answering genuinely different questions.

</details>

### ❓ Question 5: A single tiny p-value dominates

Plot the transform $\tan((0.5-p)\pi)$ against $p$ and overlay the small-$p$ approximation $1/(\pi p)$. Then, on a few toy examples, compare the Cauchy combined p-value against **Stouffer's Z** to see how each reacts to a single very strong signal buried in noise.

<details>
<summary>💡 Click here to reveal the answer!</summary>

The transform diverges like $1/(\pi p)$ as $p \to 0$, so the most significant test dominates $T$. That makes Cauchy aggregate *all* the evidence while still being driven mainly by the strongest signal — a lone tiny p-value pulls the combined p-value down hard. Stouffer, which averages on the Z-scale, instead *dilutes* that one strong signal among the other null-ish tests.

</details>

In [ ]:
# (a) the tangent transform and its small-p approximation
pp = np.logspace(-4, np.log10(0.5), 400)      # grid of p-values from 1e-4 up to 0.5 (log-spaced)
plt.loglog(pp, np.tan((0.5 - pp) * np.pi), lw=2, label=r"$\tan((0.5-p)\pi)$")
plt.loglog(pp, 1/(np.pi*pp), "--", label=r"$1/(\pi p)$ approximation")
plt.xlabel("p"); plt.ylabel("transform value")
plt.title("A small p-value dominates the Cauchy statistic")
plt.legend(); plt.tight_layout(); plt.show()

# (b) toy examples: how Cauchy vs Stouffer react to one strong signal
examples = {
    "one strong + noise": [1e-5, 0.6, 0.4, 0.8, 0.5],
    "several moderate":    [0.03, 0.04, 0.02, 0.05, 0.06],
    "all null-ish":        [0.7, 0.3, 0.9, 0.5, 0.6],
}
rows = []
for name, ps in examples.items():
    ps = np.array(ps)
    rows.append({
        "example": name,
        "Cauchy": cauchy_combination(ps),
        "Stouffer": stouffer(ps),
    })
pd.DataFrame(rows).set_index("example").round(5)

### ❓ Question 6: The payoff — correlated tests

Fisher's method (like Stouffer's) **assumes the tests are independent**. Real GWAS tests are often **correlated** (LD between SNPs, overlapping samples). What happens to the false-positive rate?

We'll simulate **null** data (no true signal) where the test statistics are *correlated*, and estimate the **type-I error rate** at $\alpha = 0.05$ for each combination method. A well-calibrated test should reject ~5% of the time.

<details>
<summary>💡 Click here to reveal the answer!</summary>

Both classic p-value *combiners* assume independence, so both **inflate** as the correlation $\rho$ grows — only Cauchy stays calibrated:

- **Fisher** ($-2\sum_k \ln p_k \sim \chi^2_{2K}$) **inflates** badly. The $\chi^2_{2K}$ null assumes $K$ *independent* tests, so positively correlated p-values make the statistic more variable than that and the type-I error climbs well above 5% (≈0.24 at $\rho=0.9$). The *effective* number of independent tests is far smaller than $K$.
- **Stouffer** inflates for the same reason — its summed Z is more variable than the standard normal it assumes (≈0.32 at $\rho=0.9$).
- **Cauchy** stays **close to 5%** across the whole range, *without being told the correlation structure* — calibrated without losing power.

That robustness to arbitrary dependence is exactly why ACAT is used for set-based and rare-variant tests — see the optional Cauchy section in [`rare_variant_Fisher.ipynb`](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/rare_variant_Fisher.ipynb), which applies the same idea to aggregate gene-level p-values.

</details>

In [ ]:
def correlated_null_pvalues(n_rep, K, rho, rng):
    # Make n_rep sets of K two-sided NULL p-values from equicorrelated N(0,1) z-scores.
    #   n_rep = number of simulated datasets,  K = tests per dataset,  rho = pairwise correlation
    cov = (1 - rho) * np.eye(K) + rho * np.ones((K, K))   # exchangeable correlation matrix
    L = np.linalg.cholesky(cov)                          # factor so that (z @ L.T) has covariance cov
    z = rng.standard_normal((n_rep, K)) @ L.T            # correlated standard-normal z-scores
    return 2 * stats.norm.sf(np.abs(z))                  # convert to two-sided p-values

def fisher(p):
    # Fisher's method: -2 * sum(log p) ~ chi-square with 2K df *under independence*.
    p = np.asarray(p, float)
    return stats.chi2.sf(-2.0 * np.sum(np.log(p)), df=2 * p.size)

n_rep = 5000                                  # number of simulated null datasets per rho
K = 20                                         # number of (correlated) tests to combine
alpha = 0.05                                   # significance threshold (nominal type-I error rate)
rhos = [0.0, 0.3, 0.6, 0.9]                    # pairwise correlations to sweep (0 = independent tests)

results = {"Fisher": [], "Stouffer": [], "Cauchy": []}   # type-I error rate per method, per rho
for rho in rhos:                               # rho = current correlation level
    P = correlated_null_pvalues(n_rep, K, rho, rng)          # (n_rep, K) matrix of null p-values
    P = np.clip(P, 1e-15, 1 - 1e-15)                         # avoid exactly 0 or 1 (numerical safety)
    fis  = np.array([fisher(row) for row in P])             # Fisher combined p-value per dataset
    stf  = np.array([stouffer(row) for row in P])           # Stouffer combined p-value per dataset
    cau  = np.array([cauchy_combination(row) for row in P]) # Cauchy combined p-value per dataset
    results["Fisher"].append((fis < alpha).mean())          # fraction of false rejections (type-I error)
    results["Stouffer"].append((stf  < alpha).mean())
    results["Cauchy"].append((cau  < alpha).mean())

x = np.arange(len(rhos)); width = 0.25         # bar positions and width for the grouped bar chart
for i, (name, vals) in enumerate(results.items()):
    plt.bar(x + (i-1)*width, vals, width, label=name)
plt.axhline(alpha, ls="--", color="k", label="nominal 0.05")
plt.xticks(x, [f"rho={r}" for r in rhos]); plt.ylabel("type-I error rate")
plt.title("Type-I error under correlated null tests (alpha = 0.05)")
plt.legend(); plt.tight_layout(); plt.show()

pd.DataFrame(results, index=[f"rho={r}" for r in rhos]).round(3)

## 🌟 4. Capstone: meta-analysing real GIANT height GWAS

The [**GIANT consortium**](https://giant-consortium.web.broadinstitute.org/index.php/GIANT_consortium_data_files) published **height** GWAS (Yengo et al. 2022) stratified into five ancestry groups: **EUR, EAS, AFR, HIS, SAS**. These are exactly the kind of data that are *meant* to be meta-analysed — and because the ancestries differ in sample size and (sometimes) effect size, they also let us see **heterogeneity** in action.

We pull a **chromosome-22 subset** of these five files (already harmonised to a common effect allele) from the course Google Drive folder and combine them per-SNP with our IVW function.

In [ ]:
# Download the prepared sumstats from the course Drive folder
!python -m pip -q install gdown
!gdown --folder "https://drive.google.com/drive/folders/1umhPkc75F39YcDQwZlQcTfbRk9maTrAW" -O /content/meta_data 2>/dev/null
import os, glob
files = sorted(glob.glob("/content/meta_data/**/*.*", recursive=True))
print(f"Downloaded {len(files)} files:", files)

In [ ]:
# Load the stratified GWAS into a dict of DataFrames keyed by stratum name.
# Expected columns (harmonised ahead of time): SNP, A1, A2, BETA, SE, N
expected_cols = {"SNP", "A1", "A2", "BETA", "SE", "N"}

def try_load(files):
    sumstats = {}
    for f in files:
        if not f.lower().endswith((".txt", ".tsv", ".csv", ".gz")):
            continue
        try:
            sep = "," if f.lower().endswith(".csv") else "\t"
            df = pd.read_csv(f, sep=sep)
            if expected_cols.issubset(df.columns):
                stem = os.path.splitext(os.path.basename(f).replace(".gz", ""))[0]
                name = stem.split("_")[-1].upper()      # e.g. giant_height_eur -> EUR
                sumstats[name] = df[list(expected_cols)]
        except Exception as e:
            print(f"  could not parse {f}: {e}")
    return sumstats

sumstats = try_load(files)

for name, df in sumstats.items():
    print(f"  {name:14s} {len(df):>6,} SNPs")

### ❓ Question 7: Per-SNP IVW meta-analysis

Harmonise the strata to a shared SNP set and meta-analyse **each SNP** across strata with IVW. (We assume the files were already allele-harmonised so A1/A2 are aligned — in a real pipeline you'd flip the sign of BETA where the effect allele is swapped.)

<details>
<summary>💡 Click here to reveal the answer!</summary>

For each SNP, stack the per-stratum BETA and SE and apply the same `ivw()` function from Section 2. Then a Z-score $\text{BETA}/\text{SE}$ gives a p-value. The meta-analysed result should have **smaller SEs** (and thus more genome-wide-significant hits) than any single stratum — the whole point of meta-analysis.

</details>

In [ ]:
# Align on the shared set of SNPs
common = set.intersection(*[set(df.SNP) for df in sumstats.values()])   # SNPs present in every stratum
strata = list(sumstats)                                                 # ordered list of stratum names
aligned = {name: df[df.SNP.isin(common)].sort_values("SNP").reset_index(drop=True)
           for name, df in sumstats.items()}

snp_ids = aligned[strata[0]].SNP.values                                # shared SNP IDs (same order everywhere)
beta_mat = np.column_stack([aligned[s].BETA.values for s in strata])   # effect sizes, shape (n_SNPs, n_strata)
se_mat   = np.column_stack([aligned[s].SE.values   for s in strata])   # standard errors, same shape

w = 1.0 / se_mat**2                            # inverse-variance weights, per SNP per stratum
beta_meta = (w * beta_mat).sum(axis=1) / w.sum(axis=1)   # IVW pooled effect, one value per SNP
se_meta   = np.sqrt(1.0 / w.sum(axis=1))       # pooled SE, per SNP
z_meta    = beta_meta / se_meta                # meta-analysis Z-score, per SNP
p_meta    = np.clip(2 * stats.norm.sf(np.abs(z_meta)), 1e-300, 1.0)   # two-sided p-value (clipped for safety)

meta = pd.DataFrame({"SNP": snp_ids, "BETA": beta_meta, "SE": se_meta,
                     "Z": z_meta, "P": p_meta})
print(f"Meta-analysed {len(meta):,} SNPs across strata: {strata}")
print(f"Mean SE per stratum: " +
      ", ".join(f"{s}={se_mat[:,i].mean():.4f}" for i,s in enumerate(strata)))
print(f"Mean SE meta-analysed: {se_meta.mean():.4f}  <-- smaller, as expected")
meta.reindex(meta.P.abs().sort_values().index).head()

### 🔭 The payoff: seeing the power you gained

The cleanest way to *feel* the benefit is to stack the five per-ancestry Manhattan plots on top of the meta-analysis. The small-sample ancestries find few or no genome-wide-significant SNPs on their own — but pooling them with IVW (bottom panel) recovers far more signal.

The real punchline is in **green**: SNPs that are **below the threshold in *every* single ancestry, yet genome-wide significant in the meta**. Those associations exist *only* because we combined the studies — two ancestries each sitting at $p\approx 6\times10^{-8}$ can combine to $p\approx 10^{-15}$. Follow a green dot up the stack: sub-threshold in every ancestry panel, then breaking through the line at the bottom.

In [ ]:
# Per-ancestry p-values (aligned to the meta), and the meta p-value from before
z_strata = beta_mat / se_mat
p_strata = np.clip(2 * stats.norm.sf(np.abs(z_strata)), 1e-300, 1.0)   # shape (n_SNPs, n_strata)

gw = 5e-8                                       # genome-wide significance threshold
x  = np.arange(len(meta))                       # SNPs ordered by chr22 position
meta_only = (p_strata > gw).all(axis=1) & (p_meta < gw)   # sig in META but in NO single ancestry

panel_names = strata + ["META"]                 # one row per ancestry, plus the meta at the bottom
panel_p     = [p_strata[:, i] for i in range(len(strata))] + [p_meta]

# median sample size per ancestry; the meta's effective N is roughly their sum
n_by = {s: int(aligned[s].N.median()) for s in strata}
n_by["META"] = sum(n_by.values())
fmt_n = lambda n: f"{n/1e6:.2f}M" if n >= 1e6 else f"{n/1e3:.0f}k"

fig, axes = plt.subplots(len(panel_names), 1, sharex=True, figsize=(8, 1.3 * len(panel_names)))
for ax, name, pv in zip(axes, panel_names, panel_p):
    is_meta = (name == "META")
    sig = pv < gw                               # genome-wide-significant SNPs in THIS panel
    ax.scatter(x[~sig], -np.log10(pv[~sig]), s=4, color="#cccccc")                       # below threshold
    ax.scatter(x[sig],  -np.log10(pv[sig]),  s=7, color="#c0392b" if is_meta else "#2c7fb8")  # GW-sig here
    ax.scatter(x[meta_only], -np.log10(pv[meta_only]), s=10, color="#27ae60", zorder=3) # the 'meta-only' SNPs
    ax.axhline(-np.log10(gw), ls="--", lw=0.8, color="r")
    ax.set_ylabel(f"{name}\nN={fmt_n(n_by[name])}", rotation=0, ha="right", va="center", fontsize=8)
    ax.text(0.995, 0.92, f"{int(sig.sum())} GW-sig", transform=ax.transAxes,
            ha="right", va="top", fontsize=8, color="#555")
    if is_meta:
        ax.set_facecolor("#fdf3f2")             # tint the meta panel
axes[-1].set_xlabel("chr22 SNPs (ordered by genomic position)")
fig.suptitle("Power gain: each ancestry alone (top) vs IVW meta-analysis (bottom)")

# legend OUTSIDE the stack explaining what the point colours mean
legend_pts = [
    plt.Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#cccccc",
               markeredgecolor="none", label="below genome-wide threshold"),
    plt.Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#2c7fb8",
               markeredgecolor="none", label="genome-wide significant in that ancestry"),
    plt.Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#c0392b",
               markeredgecolor="none", label="genome-wide significant in the meta-analysis"),
    plt.Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#27ae60",
               markeredgecolor="none", label="significant ONLY after meta-analysis"),
]
fig.legend(handles=legend_pts, loc="center left", bbox_to_anchor=(1.0, 0.5),
           frameon=False, fontsize=8, title="point colour")
fig.tight_layout()

n_only = int(meta_only.sum())
if n_only:
    k = np.where(meta_only)[0][np.argmin(p_meta[meta_only])]   # strongest 'meta-only' hit
    for ax in axes:
        ax.axvline(k, color="#27ae60", lw=0.8, ls=":", zorder=0)    # guide line across all panels
    # label the locus clearly (white box; flips to the left near the right edge so it never clips)
    ha = "left" if k < 0.8 * len(meta) else "right"
    axes[-1].annotate(meta.SNP.iloc[k], xy=(k, -np.log10(p_meta[k])),
                      xytext=(9 if ha == "left" else -9, 0), textcoords="offset points",
                      fontsize=9, fontweight="bold", color="#1e8449", ha=ha, va="center",
                      zorder=6, annotation_clip=False,
                      bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#27ae60", lw=0.8, alpha=0.95))
plt.show()

if n_only:
    per = ", ".join(f"{s} p={p_strata[k, i]:.0e}" for i, s in enumerate(strata))
    print(f"{n_only} SNPs are genome-wide significant in the META but in NO single ancestry.")
    print(f"Strongest example {meta.SNP.iloc[k]}:  {per}  ->  META p={p_meta[k]:.0e}")
else:
    print("No 'meta-only' genome-wide hits in this particular subset.")

## 🧗 5. Bonus challenges (time permitting)

### 🌟 Challenge A: Random-effects meta-analysis

Fixed effects assumes a *single* true effect. But sometimes the true effect genuinely **varies between studies** — e.g. different ancestries, environments, or measurement protocols. Then we want the **average effect across a distribution** of true effects:

$$\hat\beta_k \sim \mathrm{N}(\beta_k,\ \text{se}_k^2), \qquad \beta_k \sim \mathrm{N}(\beta,\ \tau^2).$$

Here each study now has its *own* true effect $\beta_k$ (drawn around the overall mean $\beta$), and $\hat\beta_k$ — the hat still over $\beta$ alone — is its estimate. $\tau^2$ is the **between-study heterogeneity**. The function below is provided — your job is to **read the results and build intuition**, not to re-derive the algebra:
1. **Cochran's Q** and **$I^2$** quantify heterogeneity — how big are they here?
2. The **DerSimonian–Laird** random-effects estimator inflates each variance to $w_k^{*} = 1/(\text{se}_k^2 + \hat\tau^2)$. What does that do to the weights and the pooled SE?
3. Compare the fixed- vs random-effects CI widths in the **forest plot** — which is wider, and why is that the *honest* one here?

<details>
<summary>💡 Click here to reveal the answer!</summary>

When real heterogeneity is present, fixed effects gives a CI that is **too narrow** (overconfident) — it ignores $\tau^2$. Random effects widens the CI to reflect that we're estimating the *mean* of a distribution of effects. $I^2$ is the fraction of total variation due to heterogeneity rather than sampling error; rules of thumb: ~25% low, ~50% moderate, ~75% high.

</details>

In [ ]:
# Simulate studies whose TRUE effects differ (heterogeneity), e.g. ancestry differences
beta_true, tau_true = 0.30, 0.15                # beta_true = mean effect; tau_true = between-study SD (heterogeneity)
K = 8                                           # number of studies
se = rng.uniform(0.05, 0.20, K)                 # each study's (within-study) standard error
beta_k = rng.normal(beta_true, tau_true, K)     # study-specific true effects (scattered around beta_true)
beta_hat = rng.normal(beta_k, se)               # observed estimate = true effect + sampling noise

def meta_fixed_random(beta_hat, se):
    w = 1/se**2                              # fixed-effects inverse-variance weights
    beta_fe = np.sum(w*beta_hat)/np.sum(w)   # fixed-effects pooled estimate
    se_fe   = np.sqrt(1/np.sum(w))           # fixed-effects pooled SE
    # Cochran's Q and DerSimonian-Laird tau^2
    Q  = np.sum(w * (beta_hat - beta_fe)**2) # heterogeneity statistic (weighted scatter around beta_fe)
    df = len(beta_hat) - 1                    # degrees of freedom = K - 1
    C  = np.sum(w) - np.sum(w**2)/np.sum(w)   # scaling constant in the DerSimonian-Laird formula
    tau2 = max(0.0, (Q - df) / C)             # estimated between-study variance (floored at 0)
    I2 = max(0.0, (Q - df) / Q) * 100         # % of total variation due to heterogeneity
    # random effects
    wr = 1/(se**2 + tau2)                     # random-effects weights (each variance inflated by tau^2)
    beta_re = np.sum(wr*beta_hat)/np.sum(wr)  # random-effects pooled estimate
    se_re   = np.sqrt(1/np.sum(wr))           # random-effects pooled SE
    return dict(beta_fe=beta_fe, se_fe=se_fe, beta_re=beta_re, se_re=se_re,
                Q=Q, df=df, tau2=tau2, I2=I2)

res = meta_fixed_random(beta_hat, se)
print(f"Cochran's Q = {res['Q']:.2f} (df={res['df']}),  I^2 = {res['I2']:.0f}%,  tau^2_hat = {res['tau2']:.4f}")
print(f"Fixed  effects: beta = {res['beta_fe']:.3f}  (95% CI width = {2*1.96*res['se_fe']:.3f})")
print(f"Random effects: beta = {res['beta_re']:.3f}  (95% CI width = {2*1.96*res['se_re']:.3f}  <- wider)")

In [ ]:
# Forest plot: individual studies + fixed and random pooled estimates
y = np.arange(K)[::-1]
plt.errorbar(beta_hat, y, xerr=1.96*se, fmt="s", color="#333", capsize=3, label="studies")
plt.errorbar([res['beta_fe']], [-1], xerr=[1.96*res['se_fe']], fmt="D", color="#c44",
             capsize=4, label="fixed effects")
plt.errorbar([res['beta_re']], [-2], xerr=[1.96*res['se_re']], fmt="D", color="#46c",
             capsize=4, label="random effects")
plt.axvline(beta_true, ls="--", color="grey", label="true mean")
plt.yticks(list(y)+[-1,-2], [f"Study {i+1}" for i in range(K)]+["FE pooled","RE pooled"])
plt.xlabel("effect size (95% CI)"); plt.title("Forest plot: fixed vs random effects")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

### 🌟 Challenge B: Confidence-interval coverage

A 95% confidence interval should contain the true mean effect about **95% of the time**. Does it? The plot below runs many simulated meta-analyses and draws each one's 95% CI as a horizontal line (sorted by upper edge), with the intervals that **miss** the true mean in red — fixed effects on the left, random effects on the right.

**Read the two panels and judge for yourself — don't compute anything:**
- Which method does a better job of capturing the true mean (i.e. less red)?
- Which method's intervals are *wider*, and is that a flaw or the honest thing to do?
- One method says it is "95% confident" but is wrong far more than 5% of the time. Which one — and what is it pretending not to know?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**Random effects covers close to the nominal 95%; fixed effects badly under-covers** (often only ~65%) — its red band is much bigger.

The key is that there is **no single true effect** here: each study's true effect is drawn from $\beta_k \sim \mathrm{N}(\beta, \tau^2)$, so they genuinely differ, and $\beta$ is only the *mean* of that distribution. Fixed effects assumes $\tau^2 = 0$ (one common effect) and builds its CI from the within-study SEs **alone**, ignoring the between-study variance — so its intervals are too short and miss the mean far more than 5% of the time. It is *overconfident*. Random effects adds $\tau^2$ back into the weights, widening the intervals to honestly reflect that we are estimating the **mean of a distribution** of effects. (RE can still slightly under-cover for small $K$ because $\hat\tau^2$ is itself uncertain — Hartung–Knapp adjustments help.)

</details>

In [ ]:
n_rep = 400                                   # simulated meta-analyses (kept modest so each CI line is visible)
z = 1.96                                       # 95% CI multiplier

# For each replicate, store the pooled estimate and its SE, under FE and under RE
fe_mid = np.empty(n_rep); fe_se = np.empty(n_rep)
re_mid = np.empty(n_rep); re_se = np.empty(n_rep)
for i in range(n_rep):
    bi  = rng.normal(beta_true, tau_true, K)   # fresh study-specific true effects
    se_ = rng.uniform(0.05, 0.20, K)           # fresh within-study standard errors
    bh  = rng.normal(bi, se_)                  # fresh observed estimates
    r = meta_fixed_random(bh, se_)
    fe_mid[i], fe_se[i] = r['beta_fe'], r['se_fe']
    re_mid[i], re_se[i] = r['beta_re'], r['se_re']

panels = {"Fixed effects": (fe_mid, fe_se), "Random effects": (re_mid, re_se)}

# Visualise: draw every replicate's 95% CI as one horizontal line, sorted by upper edge.
# Red lines are CIs that MISS the true mean -> the red fraction is the under-coverage.
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(10, 5))
for ax, (name, (mid, se)) in zip(axes, panels.items()):
    lo, hi = mid - z*se, mid + z*se            # the 95% CI edges
    covers = (lo <= beta_true) & (beta_true <= hi)   # does this CI contain the true mean?
    order = np.argsort(hi)                      # sort replicates by the CI's UPPER edge
    yy = np.arange(n_rep)
    c = covers[order]
    ax.hlines(yy[c],  lo[order][c],  hi[order][c],  color="#7f7f7f", alpha=0.30, lw=0.8)  # covers (grey)
    ax.hlines(yy[~c], lo[order][~c], hi[order][~c], color="#d62728", alpha=0.85, lw=1.1)  # misses (red)
    ax.axvline(beta_true, color="k", ls="--", lw=1)                                       # the true mean
    ax.set_title(name)
    ax.set_xlabel("95% CI for the pooled effect")
    ax.set_yticks([])
axes[0].set_ylabel(f"{n_rep} simulated meta-analyses\n(sorted by CI upper edge)")

legend_items = [
    plt.Line2D([0], [0], color="#7f7f7f", lw=2, label="CI covers the true mean"),
    plt.Line2D([0], [0], color="#d62728", lw=2, label="CI misses the true mean"),
    plt.Line2D([0], [0], color="k", ls="--", lw=1, label="true mean effect"),
]
fig.legend(handles=legend_items, loc="lower center", bbox_to_anchor=(0.5, -0.03),
           ncol=3, frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

## 🏁 End of the Meta-analysis Practical

You implemented, from scratch:
- **Fixed-effects IVW** and saw *empirically* why inverse-variance weights minimise the variance of the pooled estimate.
- **Stouffer's Z** and the **Cauchy combination test**, and saw why Cauchy stays calibrated under correlated tests while Stouffer and Fisher do not.
- A **real-data meta-analysis** of GIANT summary statistics.
- **Random-effects** meta-analysis and confidence-interval coverage in the bonus challenges.

🎉 Nicely done!